# Global Pollution Analysis and Energy Recovery

## Objective
The goal of this project is to analyze global pollution data to understand the relationship between pollution levels and energy recovery using the Apriori Algorithm and CNN comparison model.


In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, roc_curve, auc

from sklearn.ensemble import RandomForestClassifier

from mlxtend.frequent_patterns import apriori, association_rules

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load Dataset
df = pd.read_csv('Global_Pollution_Analysis.csv')

print(df.head())


In [ ]:
# Dataset Information
print("Dataset Shape:", df.shape)

df.info()

print(df.describe())


In [ ]:
# Check Missing Values
print(df.isnull().sum())

# Fill Numerical Missing Values
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    df[col].fillna(df[col].mean(), inplace=True)

# Fill Categorical Missing Values
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print(df.isnull().sum())


In [ ]:
# Normalize Pollution Indices
scaler = MinMaxScaler()

pollution_cols = [
    'Air_Pollution_Index',
    'Water_Pollution_Index',
    'Soil_Pollution_Index'
]

df[pollution_cols] = scaler.fit_transform(df[pollution_cols])

print(df.head())


In [ ]:
# Encode Categorical Features
le = LabelEncoder()

categorical_cols = ['Country', 'Year']

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print(df.head())


In [ ]:
# Feature Engineering - Energy Per Capita
df['Energy_Per_Capita'] = (
    df['Energy_Consumption'] / df['Population']
)

print(df.head())


In [ ]:
# Pollution Severity Categories
def pollution_category(x):
    if x < 0.33:
        return 'Low'
    elif x < 0.66:
        return 'Medium'
    else:
        return 'High'

df['Air_Pollution_Level'] = df['Air_Pollution_Index'].apply(pollution_category)
df['Water_Pollution_Level'] = df['Water_Pollution_Index'].apply(pollution_category)

print(df.head())


In [ ]:
# Pollution Trends
trend = df.groupby('Year')[[
    'Air_Pollution_Index',
    'Water_Pollution_Index',
    'Soil_Pollution_Index'
]].mean()

trend.plot(figsize=(10,6))
plt.title("Pollution Trends Over Years")
plt.ylabel("Average Pollution")
plt.show()


In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10,8))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# Distribution Plot
df[pollution_cols].hist(figsize=(12,5))
plt.show()


In [ ]:
# Prepare Data for Apriori Algorithm
basket = pd.get_dummies(df[[
    'Air_Pollution_Level',
    'Water_Pollution_Level'
]])

print(basket.head())


In [ ]:
# Apply Apriori Algorithm
frequent_itemsets = apriori(
    basket,
    min_support=0.1,
    use_colnames=True
)

print(frequent_itemsets.sort_values(by='support', ascending=False))


In [ ]:
# Generate Association Rules
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.5
)

print(rules.head())


In [ ]:
# Display Important Metrics
print(rules[[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]])


In [ ]:
# Visualize Association Rules
plt.figure(figsize=(8,6))

sns.scatterplot(
    x=rules['support'],
    y=rules['confidence'],
    size=rules['lift'],
    hue=rules['lift']
)

plt.title("Association Rules")
plt.show()


In [ ]:
# Rule Interpretation
for i in range(len(rules)):
    print("Rule:", i+1)
    print("If", rules.iloc[i]['antecedents'],
          "then", rules.iloc[i]['consequents'])
    print("Support:", rules.iloc[i]['support'])
    print("Confidence:", rules.iloc[i]['confidence'])
    print("Lift:", rules.iloc[i]['lift'])
    print()


In [ ]:
# CNN Model - Dummy Dataset
X = np.random.rand(500, 28, 28, 1)

y = np.random.randint(0, 2, 500)

y = to_categorical(y)


In [ ]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [ ]:
# Build CNN Model
model = Sequential()

model.add(Conv2D(
    32,
    (3,3),
    activation='relu',
    input_shape=(28,28,1)
))

model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(64, activation='relu'))

model.add(Dense(2, activation='softmax'))


In [ ]:
# Compile CNN Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# Train CNN Model
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)


In [ ]:
# Evaluate CNN Model
loss, accuracy = model.evaluate(X_test, y_test)

print("CNN Accuracy:", accuracy)


In [ ]:
# Confusion Matrix
y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# ROC Curve
fpr, tpr, threshold = roc_curve(
    y_true,
    y_pred[:,1]
)

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))

plt.plot(fpr, tpr, label='AUC = %0.2f' % roc_auc)

plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.title('ROC Curve')
plt.legend()

plt.show()


In [ ]:
# Cross Validation
X_ml = df[num_cols]
y_ml = np.random.randint(0,2,len(df))

rf = RandomForestClassifier()

scores = cross_val_score(
    rf,
    X_ml,
    y_ml,
    cv=5
)

print("Cross Validation Scores:", scores)
print("Average Score:", scores.mean())


In [ ]:
# Conclusion
print("Project Completed Successfully")
